In [1]:
import pandas as pd
import os

LINKS_FILE = "../data/raw/string_v12_0/4932.protein.links.full.v12.0.txt.gz"
INFO_FILE = "../data/raw/string_v12_0/4932.protein.info.v12.0.txt.gz"
OUT_FILE = "../outputs/jadon/string_v12.0_edges_tau0.7.csv"

THRESHOLD = 0.7

In [2]:
links = pd.read_csv(LINKS_FILE, sep=" ", usecols=["protein1", "protein2", "combined_score"])
print("All edges (both directions):", len(links))

links = links[links["combined_score"] >= 1000 * THRESHOLD]
print("Edges with score >= 700:", len(links))

All edges (both directions): 2824842
Edges with score >= 700: 208376


In [4]:
links = links[links["protein1"] != links["protein2"]]

a = links[["protein1", "protein2"]].min(axis=1)
b = links[["protein1", "protein2"]].max(axis=1)
links = (links.assign(protein1=a, protein2=b)
              .groupby(["protein1", "protein2"], as_index=False)["combined_score"].max())
print("Unique undirected edges:", len(links))

Unique undirected edges: 104188


In [5]:
info = pd.read_csv(INFO_FILE, sep="\t")
names = dict(zip(info.iloc[:, 0], info["preferred_name"]))

links["name1"] = links["protein1"].map(names)
links["name2"] = links["protein2"].map(names)
links["weight"] = links["combined_score"] / 1000
links.head()

,protein1,protein2,combined_score,name1,name2,weight
0,4932.Q0010,4932.Q0017,955,Q0010,Q0017,0.955
1,4932.Q0010,4932.Q0032,956,Q0010,Q0032,0.956
2,4932.Q0010,4932.Q0092,845,Q0010,Q0092,0.845
3,4932.Q0010,4932.Q0142,843,Q0010,Q0142,0.843
4,4932.Q0010,4932.Q0143,954,Q0010,Q0143,0.954


In [7]:
proteins = set(links["protein1"]) | set(links["protein2"])
hlj1 = "4932.YMR161W"

print(f"Edges kept: {len(links)}")
print(f"Proteins with at least one edge: {len(proteins)} of {len(info)}")
print(f"Isolated proteins: {len(info) - len(proteins)}")
print(f"HLJ1 degree: {((links.protein1 == hlj1) | (links.protein2 == hlj1)).sum()}")

links.to_csv(OUT_FILE, index=False)
print(f"Saved to {OUT_FILE}")

Edges kept: 104188
Proteins with at least one edge: 5791 of 6600
Isolated proteins: 809
HLJ1 degree: 25
Saved to ../outputs/jadon/string_v12.0_edges_tau0.7.csv


In [8]:
import networkx as nx

G = nx.Graph()
G.add_nodes_from(info.iloc[:, 0])
G.add_weighted_edges_from(links[["protein1", "protein2", "weight"]].itertuples(index=False))

HLJ1 = "4932.YMR161W"
SEEDS = {
    "4932.YOL013C": "HRD1",
    "4932.YLR207W": "HRD3",
    "4932.YBR201W": "DER1",
    "4932.YMR022W": "UBC7",
    "4932.YMR264W": "CUE1"
}
ALPHA = 0.85

restart = {v: (1 / len(SEEDS) if v in SEEDS else 0) for v in G}

pr_global = nx.pagerank(G, alpha=ALPHA, weight="weight")
pr_erad = nx.pagerank(G, alpha=ALPHA, weight="weight", personalization=restart)

G_del = G.copy()
G_del.remove_node(HLJ1)
pr_erad_after = nx.pagerank(
    G_del, alpha=ALPHA, weight="weight",
    personalization={v: restart[v] for v in G_del}
)

pr = pd.DataFrame({
    "name": [names.get(v, v) for v in G],
    "degree": [G.degree(v) for v in G],
    "pagerank_global": [pr_global[v] for v in G],
    "pagerank_erad": [pr_erad[v] for v in G],
    "pagerank_erad_after": [pr_erad_after.get(v, float("nan")) for v in G],
}, index=list(G))

pr["rank_before"] = pr["pagerank_erad"].rank(ascending=False)
pr["rank_after"] = pr["pagerank_erad_after"].rank(ascending=False)
pr["rank_change"] = pr["rank_before"] - pr["rank_after"]
pr["is_seed"] = pr.index.isin(list(SEEDS) + [HLJ1])

pr.to_csv("pagerank_tau0.7.csv")

print("Sums:", round(sum(pr_global.values()), 6), round(sum(pr_erad.values()), 6))

cand = pr[~pr["is_seed"]].sort_values("pagerank_erad", ascending=False)
print("\nTop 20 non-seed proteins by ERAD PageRank:")
print(cand.head(20)[[
    "name", "degree", "pagerank_erad",
    "rank_before", "rank_after", "rank_change"
]].to_string())

print("\nBiggest drops after deleting HLJ1:")
print(cand.sort_values("rank_change").head(10)[["name", "degree", "rank_before", "rank_after", "rank_change"]].to_string())

print("\nHLJ1, KAR2, YDJ1:")
print(pr.loc[[HLJ1, "4932.YJL034W", "4932.YNL064C"], ["name", "pagerank_erad", "rank_before", "rank_after"]].to_string())

Sums: 1.0 1.0

Top 20 non-seed proteins by ERAD PageRank:
               name  degree  pagerank_erad  rank_before  rank_after  rank_change
4932.YDL126C  CDC48     106       0.012179          6.0         6.0          0.0
4932.YGR048W   UFD1      57       0.011089          7.0         7.0          0.0
4932.YIL030C   SSM4      34       0.011046          8.0         8.0          0.0
4932.YML013W   UBX2      31       0.011024          9.0         9.0          0.0
4932.YBR170C   NPL4      73       0.010977         10.0        10.0          0.0
4932.YDR057W   YOS9      21       0.010366         11.0        11.0          0.0
4932.YLR378C  SEC61     215       0.010190         12.0        12.0          0.0
4932.YML029W   USA1      17       0.010032         13.0        13.0          0.0
4932.YDR411C   DFM1      19       0.009776         14.0        14.0          0.0
4932.YJL034W   KAR2      94       0.009689         15.0        15.0          0.0
4932.YMR297W   PRC1      45       0.007987         

In [9]:
import random, time

N_SOURCES = 500
random.seed(0)
candidates = [v for v in G if v != HLJ1 and G.degree(v) > 0]
sources = random.sample(candidates, N_SOURCES)
targets = [v for v in G if v != HLJ1]

t = time.time()
bt_before = nx.betweenness_centrality_subset(G, sources, targets, normalized=False)
bt_after = nx.betweenness_centrality_subset(G_del, sources, targets, normalized=False)
print(f"Done in {time.time() - t:.0f} s")

bt = pd.DataFrame({
    "name": [names.get(v, v) for v in G],
    "degree": [G.degree(v) for v in G],
    "hlj1_neighbour": [G.has_edge(v, HLJ1) for v in G],
    "bt_before": [bt_before[v] for v in G],
    "bt_after": [bt_after.get(v, float("nan")) for v in G],
}, index=list(G))
bt["bt_change"] = bt["bt_after"] - bt["bt_before"]
bt["is_seed"] = bt.index.isin(list(SEEDS) + [HLJ1])
bt.to_csv("betweenness_tau0.7.csv")

print(f"\nHLJ1's own betweenness (traffic it carried): {bt_before[HLJ1]:.1f}")

cand = bt[~bt["is_seed"]]
print("\nBiggest GAINS after deleting HLJ1 (took over its traffic):")
print(cand.sort_values("bt_change", ascending=False).head(15)[["name", "degree", "hlj1_neighbour", "bt_before", "bt_after", "bt_change"]].to_string())

print("\nBiggest LOSSES after deleting HLJ1:")
print(cand.sort_values("bt_change").head(10)[["name", "degree", "hlj1_neighbour", "bt_before", "bt_after", "bt_change"]].to_string())

print(f"\nProteins whose betweenness changed at all: {(cand['bt_change'].abs() > 1e-9).sum()}")

Done in 27 s

HLJ1's own betweenness (traffic it carried): 32.5

Biggest GAINS after deleting HLJ1 (took over its traffic):
                 name  degree  hlj1_neighbour     bt_before      bt_after  bt_change
4932.YNL064C     YDJ1      67           False   2710.514435   2714.573612   4.059178
4932.YDL126C    CDC48     106           False  12351.545058  12355.074054   3.528996
4932.YLL039C     UBI4     294           False  27230.051990  27232.550823   2.498832
4932.YNL007C     SIS1     113           False   2383.266212   2385.571856   2.305644
4932.YDR320C     SWA2      46            True   1835.808204   1837.445541   1.637337
4932.YMR214W     SCJ1      40           False   1289.174775   1290.190568   1.015793
4932.YKL173W   SNU114     237           False  12112.993880  12113.972174   0.978294
4932.YNL172W     APC1      46           False   2746.504510   2747.442775   0.938265
4932.YEL037C    RAD23     115           False   7647.610763   7648.433181   0.822418
4932.YGR184C     UBR1     

In [10]:
d_before = nx.multi_source_dijkstra_path_length(G, list(SEEDS), weight=lambda u, v, d: 1)
d_after = nx.multi_source_dijkstra_path_length(G_del, list(SEEDS), weight=lambda u, v, d: 1)
d_hlj1 = nx.single_source_shortest_path_length(G, HLJ1)

dist = pd.DataFrame({
    "d_seed": [d_before.get(v, float("inf")) for v in G],
    "d_seed_after": [d_after.get(v, float("inf")) for v in G],
    "d_hlj1": [d_hlj1.get(v, float("inf")) for v in G],
}, index=list(G))
dist["d_change"] = dist["d_seed_after"] - dist["d_seed"]

print(dist["d_seed"].value_counts().sort_index().head(8))
print("Proteins whose distance to the seeds changed:", (dist["d_change"].fillna(0) != 0).sum())

d_seed
0.0       5
1.0      49
2.0     950
3.0    2940
4.0    1245
5.0     359
6.0     122
7.0      34
Name: count, dtype: int64
Proteins whose distance to the seeds changed: 1


In [11]:
seed_nbrs = {s: set(G[s]) for s in SEEDS}

def jaccard(a_nbrs, b_nbrs):
    union = a_nbrs | b_nbrs
    return len(a_nbrs & b_nbrs) / len(union) if union else 0.0

rows = {}
for v in G:
    nv = set(G[v])
    js = [jaccard(nv, seed_nbrs[s]) for s in SEEDS if s != v]
    rows[v] = {"J_mean": sum(js) / len(js), "J_max": max(js)}
jac = pd.DataFrame.from_dict(rows, orient="index")

print(jac.sort_values("J_mean", ascending=False).head(10).assign(name=lambda d: d.index.map(names)))

                J_mean     J_max  name
4932.YML029W  0.536519  0.695652  USA1
4932.YLR207W  0.491518  0.629630  HRD3
4932.YBR201W  0.482029  0.629630  DER1
4932.YDR057W  0.461286  0.592593  YOS9
4932.YOL013C  0.428394  0.487179  HRD1
4932.YDR411C  0.412676  0.464286  DFM1
4932.YMR022W  0.402841  0.437500  UBC7
4932.YMR264W  0.391372  0.454545  CUE1
4932.YIL030C  0.380422  0.458333  SSM4
4932.YML013W  0.323339  0.394737  UBX2


In [12]:
from collections import Counter
N_RUNS = 20
HRD1 = "4932.YOL013C"

def louvain_runs(graph):
    return [{v: i for i, c in enumerate(nx.community.louvain_communities(graph, weight="weight", seed=s)) for v in c}
            for s in range(N_RUNS)]

def frac_with(parts, target):
    counts = Counter()
    for p in parts:
        counts.update(v for v, c in p.items() if c == p[target])
    return {v: k / len(parts) for v, k in counts.items()}

parts_before = louvain_runs(G)
parts_after = louvain_runs(G_del)

with_hlj1 = frac_with(parts_before, HLJ1)
with_erad = frac_with(parts_before, HRD1)
with_erad_after = frac_with(parts_after, HRD1)

comm = pd.DataFrame({
    "with_HLJ1": [with_hlj1.get(v, 0) for v in G],
    "with_ERAD": [with_erad.get(v, 0) for v in G],
    "with_ERAD_after": [with_erad_after.get(v, 0) for v in G],
}, index=list(G))
comm["ERAD_switch"] = comm["with_ERAD_after"] - comm["with_ERAD"]

print("Seeds sharing HLJ1's community:", {SEEDS[s]: with_hlj1.get(s, 0) for s in SEEDS})
print(
    "Proteins that switched ERAD membership by >= 50% of runs:",
    (comm["ERAD_switch"].abs() >= 0.5).sum()
)

Seeds sharing HLJ1's community: {'HRD1': 0.2, 'HRD3': 0.2, 'DER1': 0.2, 'UBC7': 0.2, 'CUE1': 0.2}
Proteins that switched ERAD membership by >= 50% of runs: 0


In [14]:
ESSENTIAL_FILE = "../data/raw/essential_proteins.csv"
essential = {"4932." + orf for orf in pd.read_csv(ESSENTIAL_FILE, header=None)[1]}

pr["pct_change"] = (pr["pagerank_erad_after"] - pr["pagerank_erad"]) / pr["pagerank_erad"] * 100
bt["bt_pct_change"] = (bt["bt_change"] / bt["bt_before"] * 100).where(bt["bt_before"] > 0, 0)

allm = (pr[["name", "degree", "is_seed", "pagerank_erad", "pct_change"]]
        .join(bt[["bt_before", "bt_pct_change"]])
        .join(dist).join(jac).join(comm))
allm["hlj1_neighbour"] = [G.has_edge(v, HLJ1) for v in allm.index]
allm["essential"] = allm.index.isin(essential)
allm.to_csv("all_metrics_tau0.7.csv")

READOUTS = {"4932.YJL034W": "KAR2", "4932.YNL064C": "YDJ1"}
pool = allm[~allm["is_seed"] & ~allm.index.isin(READOUTS) & (allm["d_seed"] <= 2)].copy()
print(f"Candidate pool (within 2 hops of a seed): {len(pool)}")

top10 = lambda col: pool[col] >= pool[col].quantile(0.9)
flags = pd.DataFrame(index=pool.index)
flags["ERAD PageRank"] = top10("pagerank_erad")
flags["Jaccard"] = top10("J_max")
flags["Bridge"] = top10("bt_before")
flags["Community"] = (pool["with_HLJ1"] >= 0.8) | (pool["with_ERAD"] >= 0.8)
flags["PageRank drop"] = pool["pct_change"] <= -1
flags["Betweenness change"] = pool["bt_pct_change"].abs() >= 1
flags["Distance change"] = pool["d_change"] > 0

RELEVANT = ["ERAD PageRank", "Jaccard", "Bridge", "Community"]
AFFECTED = ["PageRank drop", "Betweenness change", "Distance change"]
pool["n_relevant"] = flags[RELEVANT].sum(axis=1)
pool["n_affected"] = flags[AFFECTED].sum(axis=1)
pool["reasons"] = flags.apply(lambda r: ", ".join(r.index[r]), axis=1)

affected = pool[(pool["n_affected"] >= 1) & (pool["n_relevant"] >= 1)]
relevant = pool[(pool["n_affected"] == 0) & (pool["n_relevant"] >= 2)]
print(f"Affected by deletion + relevant: {len(affected)}")
print(f"Strongly relevant (2+ relevance flags): {len(relevant)}")

N_FINAL = 20
affected = affected.sort_values(["n_affected", "pct_change"], ascending=[False, True])
relevant = relevant.sort_values(["n_relevant", "pagerank_erad"], ascending=False)
final = pd.concat([
    affected.assign(group="Affected by HLJ1 deletion"),
    relevant.assign(group="ERAD core")
]).head(N_FINAL)

cols = ["name", "group", "degree", "d_seed", "hlj1_neighbour", "essential", "pct_change", "reasons"]
final[cols].to_csv("final_candidates_tau0.7.csv")
print(final[cols].round(2).to_string())

print("\nReference proteins already chosen by BCMB:")
print(allm.loc[list(READOUTS), ["name", "d_seed", "pagerank_erad", "pct_change", "J_max", "with_HLJ1", "with_ERAD"]].round(4).to_string())

Candidate pool (within 2 hops of a seed): 996
Affected by deletion + relevant: 19
Strongly relevant (2+ relevance flags): 85
               name                      group  degree  d_seed  hlj1_neighbour  essential  pct_change                                                reasons
4932.YJL162C   JJJ2  Affected by HLJ1 deletion      31     2.0           False      False       -1.57           Community, PageRank drop, Betweenness change
4932.YFR041C   ERJ5  Affected by HLJ1 deletion      22     2.0            True      False       -8.35                               Community, PageRank drop
4932.YER103W   SSA4  Affected by HLJ1 deletion      36     2.0            True      False       -7.86                               Community, PageRank drop
4932.YDL229W   SSB1  Affected by HLJ1 deletion      41     2.0            True      False       -6.57                               Community, PageRank drop
4932.YGL018C   JAC1  Affected by HLJ1 deletion      45     2.0            True       True 